# 05 — Timbre Features (Colab + Drive)

Writes `features/timbre/timbre_song.csv` on Drive. GPU Off. Needs 00+01.


## Colab + Drive (every notebook)

1. Open in **Google Colab**.
2. Run **Mount Drive** and click **Allow**.
3. Shared folder: `/content/drive/MyDrive/MTG_Instrument`
4. GPU **On** only for 02, 03, 07. Off for 00, 01, 04–06, 09.
5. Do **not** re-download mels after notebook 00.


In [ ]:
!pip install -q librosa soundfile tqdm


## Mount Drive


In [ ]:
from pathlib import Path
import os

DRIVE_ROOT = Path("/content/drive/MyDrive/MTG_Instrument")

if not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")
else:
    print("Drive already mounted")

for sub in ["dataset/logmel_songs", "annotations", "features", "checkpoints", "results"]:
    (DRIVE_ROOT / sub).mkdir(parents=True, exist_ok=True)

os.environ["MTG_ROOT"] = str(DRIVE_ROOT)
print("Drive ready:", DRIVE_ROOT)


In [ ]:
from pathlib import Path
import os, json, random, re, shutil, socket, urllib.request
import numpy as np
import pandas as pd

DRIVE_ROOT = Path(os.environ.get("MTG_ROOT", "/content/drive/MyDrive/MTG_Instrument"))
ROOT = DRIVE_ROOT
MEL_DIR = ROOT / "dataset" / "logmel_songs"
ANN_DIR = ROOT / "annotations"
FEAT_DIR = ROOT / "features"
CKPT_DIR = ROOT / "checkpoints"
RESULTS_DIR = ROOT / "results"
MANIFEST = ROOT / "dataset" / "song_manifest.csv"
RAW_ANN = "https://raw.githubusercontent.com/MTG/mtg-jamendo-dataset/master/data"
NEEDED_ANN = [
    "splits/split-0/autotagging_genre-train.tsv",
    "splits/split-0/autotagging_genre-validation.tsv",
    "splits/split-0/autotagging_genre-test.tsv",
    "splits/split-0/autotagging_instrument-train.tsv",
    "splits/split-0/autotagging_instrument-validation.tsv",
    "splits/split-0/autotagging_instrument-test.tsv",
    "autotagging_genre.tsv",
    "autotagging_instrument.tsv",
]
SEED = 42
random.seed(SEED)
np.random.seed(SEED)


def check_internet(host="github.com", port=443, timeout=5) -> bool:
    try:
        socket.create_connection((host, port), timeout=timeout).close()
        return True
    except OSError:
        return False


def normalize_track_id(raw) -> str | None:
    m = re.search(r"(\d+)", str(raw))
    return f"{int(m.group(1)):07d}" if m else None


def ensure_annotations():
    dest_train = ANN_DIR / "splits" / "split-0" / "autotagging_genre-train.tsv"
    if dest_train.exists():
        return
    if not check_internet():
        raise FileNotFoundError("Split TSVs missing and no Internet. Enable Internet and re-run.")
    print("Downloading annotation TSVs to Drive...")
    for rel in NEEDED_ANN:
        dest = ANN_DIR / rel
        dest.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(f"{RAW_ANN}/{rel}", dest)
        print(" ", dest)


def load_split_ids(split: str, subset: str = "genre") -> set[str]:
    """First column only — extra tag tabs break pandas read_csv."""
    name = f"autotagging_{subset}-{split}.tsv"
    path = ANN_DIR / "splits" / "split-0" / name
    if not path.exists():
        path = ANN_DIR / name
    if not path.exists():
        raise FileNotFoundError(path)
    ids = set()
    with open(path, encoding="utf-8", errors="replace") as f:
        f.readline()
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            tid = normalize_track_id(line.split("\t")[0])
            if tid:
                ids.add(tid)
    print(f"{split:12s} {len(ids):6d} ids ← {path}")
    return ids


def iter_tsv_rows(path: Path):
    """Yield dict with TRACK_ID and remaining fields joined as TAGS."""
    with open(path, encoding="utf-8", errors="replace") as f:
        header = f.readline().strip().split("\t")
        for line in f:
            parts = line.rstrip("\n").split("\t")
            if not parts:
                continue
            row = {"TRACK_ID": parts[0]}
            if len(parts) >= 6:
                row["TAGS"] = "\t".join(parts[5:])
            elif len(parts) > 1:
                row["TAGS"] = parts[-1]
            else:
                row["TAGS"] = ""
            yield row


ensure_annotations()
print("ROOT   ", ROOT)
print("MEL_DIR", MEL_DIR, "npy=", len(list(MEL_DIR.rglob("*.npy"))))
print("ANN_DIR", ANN_DIR)
print("MANIFEST", MANIFEST, "exists=", MANIFEST.exists())


## Extract


In [ ]:
import librosa
from tqdm.auto import tqdm
if not MANIFEST.exists():
    raise FileNotFoundError("Run 01 first")
manifest = pd.read_csv(MANIFEST)
manifest["song_id"] = manifest["song_id"].astype(str).map(lambda s: normalize_track_id(s) or s)
WINDOW_SEC, SR = 15.0, 22050

def mel_proxy_features(S):
    freqs = np.linspace(0, 1, S.shape[0])[:, None]
    return dict(
        spectral_centroid_mean=float((S*freqs).sum()/(S.sum()+1e-6)),
        spectral_bandwidth_mean=float(np.std(S.mean(1))),
        spectral_contrast_mean=float(S.max(0).mean()-S.min(0).mean()),
        spectral_flatness_mean=float(np.exp(np.mean(np.log(S+1e-6)))/(np.mean(S)+1e-6)),
        rms_mean=float(np.sqrt(np.mean(S**2))),
        spectral_flux_mean=float(np.mean(np.abs(np.diff(S.mean(0))))),
    )


rows = []
for _, rec in tqdm(manifest.iterrows(), total=len(manifest)):
    sid = str(rec["song_id"])
    try:
        S = np.load(rec["mel_abs"])
        if S.ndim == 3:
            S = S.mean(0)
        feat = mel_proxy_features(S)
        feat.update({"song_id": sid, "source": "mel_proxy", "split": rec["split"]})
        rows.append(feat)
    except Exception as e:
        print("fail", sid, e)
df = pd.DataFrame(rows)
out = FEAT_DIR / "timbre"
out.mkdir(parents=True, exist_ok=True)
df.to_csv(out / "timbre_song.csv", index=False)
print("wrote", out, len(df))
